# Task 3: autonomous financial agents

See `specs/task3-agentic.md`. Tool order is selected by the model from observations; Analyst and Writer have enforced separate registries. Tests use fixtures; live traces below are separate.

In [1]:
from pathlib import Path
import os, sys, subprocess, zipfile

if not Path("pyproject.toml").exists():
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Run locally from repository root or upload assessment_code.zip in Colab")
    print("Upload assessment_code.zip from the project's dist folder.")
    uploads = files.upload()
    bundle = next((name for name in uploads if name.endswith(".zip")), None)
    if bundle is None:
        raise RuntimeError("Code bundle ZIP required")
    target = Path("/content/cdazzdev_assessment").resolve()
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle) as archive:
        for member in archive.infolist():
            destination = (target / member.filename).resolve()
            if not destination.is_relative_to(target):
                raise ValueError("Unsafe ZIP member")
        archive.extractall(target)
    os.chdir(target)
from importlib.metadata import version, PackageNotFoundError
try:
    version("cdazzdev-mle-assessment")
except PackageNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"])
print("Workspace:", Path.cwd())

Workspace: C:\ZUshan\CDAZZDEV — ML Technical Assessment


In [2]:
import os
try:
    from google.colab import userdata
    for name in ["GROQ_API_KEY", "HF_TOKEN"]:
        try:
            value = userdata.get(name)
            if value:
                os.environ[name] = value
        except Exception:
            print(name, "not configured or notebook access disabled")
except ImportError:
    print("Local run uses environment variables.")
from shared.preflight import check
print(check(inference=bool(os.getenv("GROQ_API_KEY"))))

Local run uses environment variables.


{'python': '3.13.7', 'groq_key_configured': False, 'hf_token_configured': False, 'inference_model': 'openai/gpt-oss-20b', 'packages': {'numpy': '2.5.3', 'pandas': '2.3.3', 'pydantic': '2.13.5', 'yfinance': '0.2.66', 'langgraph': '1.2.14'}}


In [3]:
TICKER = "AAPL"
RUN_LIVE = bool(os.getenv("GROQ_API_KEY"))
print("Live agent enabled:", RUN_LIVE)

Live agent enabled: False


## Control-flow checks
Tests prove permissions, observed failure -> fallback, typed clarification, memory reuse, cache, and JSONL format. Temporary fixture logs are not submitted as live traces.

In [4]:
subprocess.run([sys.executable, "-m", "pytest", "tests/test_agents.py"], check=True)

CompletedProcess(args=['C:\\ZUshan\\CDAZZDEV — ML Technical Assessment\\.venv\\Scripts\\python.exe', '-m', 'pytest', 'tests/test_agents.py'], returncode=0)

## Single-agent autonomous research

In [5]:
from task3_agentic.engine import ResearchSystem
import json
if RUN_LIVE:
    single = ResearchSystem()
    single_result = single.run(TICKER, mode="single", refresh=True)
    for message in single.messages:
        print(json.dumps(message, indent=2))
    print(json.dumps(single_result["report"], indent=2))
else:
    print("NOT EXECUTED: configure GROQ_API_KEY for live tool/observe/replan evidence")

NOT EXECUTED: configure GROQ_API_KEY for live tool/observe/replan evidence


## Restricted two-agent pipeline and critique loop

In [6]:
if RUN_LIVE:
    multi = ResearchSystem()
    multi_result = multi.run(TICKER, mode="multi", refresh=True)
    for message in multi.messages:
        print(json.dumps(message, indent=2))
    print(json.dumps(multi_result["report"], indent=2))
    assert multi.clarification_count == 1, "Required critique loop did not execute"

## Short-term memory: no tool refetch

In [7]:
if RUN_LIVE:
    before = multi.tool_calls
    memory_answer = multi.follow_up("What quantitative risk measure did you already retrieve? Use only existing evidence.")
    print(json.dumps(memory_answer, indent=2))
    assert multi.tool_calls == before

## Persistent cache: second session uses saved report

In [8]:
if RUN_LIVE:
    cached = ResearchSystem()
    cached_result = cached.run(TICKER, mode="multi")
    print("Cache hit:", cached_result["cache_hit"], "New tool calls:", cached_result["new_tool_calls"])
    assert cached_result["cache_hit"] and cached_result["new_tool_calls"] == 0

## Controlled live failure recovery
This separate run injects a single search failure, then exposes real tools. Inspect the trace for an alternative decision. It is explicitly labeled failure injection, not an organic source outage.

In [9]:
from task3_agentic.tools import FinancialTools
from shared.llm import JSONClient
class OneSearchFailure(FinancialTools):
    def __init__(self, client):
        super().__init__(client)
        self.failed = False
    def web_search(self, query):
        if not self.failed:
            self.failed = True
            raise RuntimeError("Controlled one-time search failure")
        return super().web_search(query)
if RUN_LIVE:
    client = JSONClient()
    failure_tools = OneSearchFailure(client)
    failure_system = ResearchSystem(client, failure_tools, log_dir="task3_agentic/logs/failure_demo", cache_dir="task3_agentic/cache/failure_demo")
    failure_result = failure_system.run(TICKER, mode="single", refresh=True)
    for message in failure_system.messages:
        print(json.dumps(message, indent=2))
    print("Injected failure executed:", failure_tools.failed)
    assert failure_tools.failed, "Agent did not select web_search; this run does not prove injected recovery"

## Actual observability artifact

In [10]:
from pathlib import Path
path = Path("task3_agentic/logs/agent_trace.jsonl")
if path.exists():
    entries = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    for entry in entries:
        print(json.dumps(entry))
    assert all(len(entry["output"]) <= 200 for entry in entries)
else:
    print("NOT GENERATED: no live tool calls have run; Task 3 evidence incomplete")

NOT GENERATED: no live tool calls have run; Task 3 evidence incomplete


## Candidate interpretation
Explain actual tool choices, failure alternatives, enforced restrictions, typed headlines passed to Analyst, clarification incorporation, cache freshness, and the distinction between search snippets and verified source facts. Download the notebook and actual logs with outputs retained.